In [ ]:
import json
import pandas as pd
from google.colab import files

drustvene_igre_putanja = 'BoardGames_ndjson.json'

drustvene_igre = []
with open(drustvene_igre_putanja, 'r', encoding='utf-8') as f:
    for linija in f:
        linija = linija.strip()
        if linija:
            drustvene_igre.append(json.loads(linija))

if isinstance(drustvene_igre, list) and len(drustvene_igre) > 0 and isinstance(drustvene_igre[0], dict):
    broj_kolona = len(drustvene_igre[0].keys())
else:
    print("Struktura fajla nije lista objekata.")

df = pd.DataFrame(drustvene_igre)

print("Broj redova i kolona na samom pocetku ciscenja: ", df.shape)     # (broj_redova, broj_kolona)
print("Nazivi kolona: ", df.columns)          # nazivi kolona
print("Tip podataka: ", df.info())            # tip podataka svake kolone + broj non-null vrednosti
print("Statistika brojcanih obelezja: ", df.describe())

#roj jedinstvenih vrednosti i procenat null vrednosti po koloni
unique_counts = df.nunique()
null_percentage = df.isnull().mean() * 100

rezultat = pd.DataFrame({
    "broj_jedinstvenih": unique_counts,
    "procenat_null": null_percentage
})

#drop sva obelezja iz df koja su vise od 60% null
kolone_za_drop = rezultat[rezultat['procenat_null'] > 30].index.tolist()
df = df.drop(columns=kolone_za_drop)

print("Broj kolona posle dropa:", df.shape[1])
#print("Nazivi kolona novih:", df.columns.tolist())

#zameni sve vrednosti max igraca
def max_100(vrednost):
    if pd.isna(vrednost):
        return vrednost
    numbers = []
    if isinstance(vrednost, str):
        vrednost = vrednost.replace('–','-')
        delovi = vrednost.split(',')
        for part in delovi:
            part = part.strip()
            if '-' in part:
                start, end = map(int, part.split('-'))
                numbers.extend(range(start, end+1))
            else:
                numbers.append(int(part))
    else:
        numbers.append(int(vrednost))

    numbers = [min(x, 100) for x in numbers]
    return ','.join(map(str, numbers))

df['details.maxplayers'] = df['details.maxplayers'].apply(max_100)
print("Prikaz unique max igraca:", df['details.maxplayers'].unique())

def minplayers_max10(vrednost):
    if pd.isna(vrednost):
        return vrednost
    numbers = []
    if isinstance(vrednost, str):
        vrednost = vrednost.replace('–','-')
        delovi = vrednost.split(',')
        for part in delovi:
            part = part.strip()
            if '-' in part:
                start, end = map(int, part.split('-'))
                numbers.extend([min(x, 10) for x in range(start, end+1)])
            else:
                numbers.append(min(int(part), 10))
    else:
        numbers.append(min(int(vrednost), 10))
    return ','.join(map(str, numbers))

df = df.dropna(subset=['details.minplayers'])
df['details.minplayers'] = df['details.minplayers'].apply(minplayers_max10)
print("Prikaz unique min igraca:", df['details.minplayers'].unique())

#drop ostalih useless kolona:
kol_za_drop = [
    "details.image",
    "details.thumbnail",
    "attributes.total"
]
df = df.drop(columns=kol_za_drop, errors="ignore")
print("Broj kolona posle drugog dropa:", df.shape[1])

#zbog puno 0 vrednosti koje mogu da kvare upite, stavljeni su na null
df["details.minplayers"] = pd.to_numeric(df["details.minplayers"], errors='coerce')
df["details.maxplayers"] = pd.to_numeric(df["details.maxplayers"], errors='coerce')

df["details.minplayers"] = df["details.minplayers"].replace(0, pd.NA)
df["details.maxplayers"] = df["details.maxplayers"].replace(0, pd.NA)
print("Null vrednosti u details.minplayers:", df["details.minplayers"].isna().sum())
print("Null vrednosti u details.maxplayers:", df["details.maxplayers"].isna().sum())

print(df['attributes.boardgamecategory'].unique())

df['categories'] = df['attributes.boardgamecategory'].apply(lambda x: x.split(',') if pd.notnull(x) else [])
print(df.shape)

#preimenovanje kolona zbog tacke u nazivu:
df = df.rename(columns={
    "attributes.boardgamecategory": "category",
    "attributes.boardgamedesigner": "designer",
    "attributes.boardgamemechanic": "mechanic",
    "attributes.boardgamepublisher": "publisher",
    "details.description": "description",
    "details.maxplayers": "maxplayers",
    "details.maxplaytime": "maxplaytime",
    "details.minage": "minage",
    "details.minplayers": "minplayers",
    "details.minplaytime": "minplaytime",
    "details.name": "name",
    "details.playingtime": "playingtime",
    "details.yearpublished": "yearpublished",
    "game.id": "game_id",
    "game.type": "game_type",
    "row_names": "row_names",
    "stats.average": "average",
    "stats.averageweight": "averageweight",
    "stats.bayesaverage": "bayesaverage",
    "stats.median": "median",
    "stats.numcomments": "numcomments",
    "stats.numweights": "numweights",
    "stats.owned": "owned",
    "stats.stddev": "stddev",
    "stats.trading": "trading",
    "stats.usersrated": "usersrated",
    "stats.wanting": "wanting",
    "stats.wishing": "wishing",
    "categories": "categories"
})

#ciscenje null u category zbog indeksa
df = df.dropna(subset=['category'])
print(df['category'].isnull().sum())
print("Broj redova i kolona na kraju ciscenja", df.shape)

print(df.info())

df.to_json("drustvene_igre_clean.json", orient="records", lines=True, force_ascii=False)
files.download("drustvene_igre_clean.json")

print(df['description'].head)

Broj redova i kolona na samom pocetku ciscenja:  (90400, 81)
Nazivi kolona:  Index(['attributes.boardgameartist', 'attributes.boardgamecategory',
       'attributes.boardgamecompilation', 'attributes.boardgamedesigner',
       'attributes.boardgameexpansion', 'attributes.boardgamefamily',
       'attributes.boardgameimplementation', 'attributes.boardgameintegration',
       'attributes.boardgamemechanic', 'attributes.boardgamepublisher',
       'attributes.t.links.concat.2....', 'attributes.total',
       'details.description', 'details.image', 'details.maxplayers',
       'details.maxplaytime', 'details.minage', 'details.minplayers',
       'details.minplaytime', 'details.name', 'details.playingtime',
       'details.thumbnail', 'details.yearpublished', 'game.id', 'game.type',
       'polls.language_dependence', 'polls.suggested_numplayers.1',
       'polls.suggested_numplayers.10', 'polls.suggested_numplayers.2',
       'polls.suggested_numplayers.3', 'polls.suggested_numplayers.4',


In [ ]:
from google.colab import files

uploaded = files.upload()

def process_game(game):
    stats = {
        'Average': game.pop('average', None),
        'BayesAverage': game.pop('bayesaverage', None),
        'Median': game.pop('median', None),
        'StdDev': game.pop('stddev', None),
        'AverageWeight': game.pop('averageweight', None),
        'NumComments': game.pop('numcomments', None),
        'NumWeights': game.pop('numweights', None),
        'UsersRated': game.pop('usersrated', None),
        'Owned': game.pop('owned', None),
        'Trading': game.pop('trading', None),
        'Wanting': game.pop('wanting', None),
        'Wishing': game.pop('wishing', None)
    }
    players = {
        'MinPlayers': game.pop('minplayers', None),
        'MaxPlayers': game.pop('maxplayers', None),
        'MinPlayTime': game.pop('minplaytime', None),
        'MaxPlayTime': game.pop('maxplaytime', None),
        'PlayingTime': game.pop('playingtime', None),
        'MinAge': game.pop('minage', None)
    }
    design = {
        'Designer': game.pop('designer', None),
        'Mechanic': game.pop('mechanic', None),
        'Publisher': game.pop('publisher', None),
        'Category': game.pop('category', None),
        'Categories': game.pop('categories', None)
    }
    new_game = {
        'GameId': game.pop('game_id', None),
        'Name': game.pop('name', None),
        'Description': game.pop('description', None),
        'YearPublished': game.pop('yearpublished', None),
        'Type': game.pop('game_type', None),
        'Stats': stats,
        'Players': players,
        'Design': design
    }
    return new_game

def process_and_write_bucket(bucket, output_file):
    restructured_games = []
    for game in bucket:
        restructured_game = process_game(game)
        restructured_games.append(restructured_game)

    with open(output_file, 'a', encoding='utf-8') as file:
        for game in restructured_games:
            json.dump(game, file)
            file.write('\n')

def read_in_buckets(file_path, bucket_size):
    with open(file_path, 'r', encoding='utf-8') as file:
        bucket = []
        for line in file:
            game = json.loads(line)
            bucket.append(game)
            if len(bucket) >= bucket_size:
                yield bucket
                bucket = []
        if bucket:
            yield bucket

drustvene_igre_putanja = '/content/drustvene_igre_clean.json'
output_file = '/content/baza_igre_restructured.json'

bucket_size = 20000

for bucket in read_in_buckets(drustvene_igre_putanja, bucket_size):
    process_and_write_bucket(bucket, output_file)

print("Podaci su uspešno prestrukturirani i sačuvani u novoj JSON datoteci.")

Saving drustvene_igre_clean.json to drustvene_igre_clean.json
Podaci su uspešno prestrukturirani i sačuvani u novoj JSON datoteci.
